# 05 — Model Provenance Verification, from scratch

Companion notebook to `../06-data-compliance-and-model-governance.md`.

This notebook implements the checksum/provenance-verification pattern Chapter 6 describes for
self-hosted open-weight models: before any downloaded model artifact is deployed to a SageMaker
endpoint or batch job, its cryptographic hash is checked against a reference hash the publisher
(DeepSeek, in this project's case) published for that exact release. A mismatch is a hard stop, not a
warning.

Two cases are demonstrated: a clean, genuine download (hash matches, deployment proceeds), and a
tampered checkpoint -- a single byte altered somewhere in a large artifact -- which the hash check
catches and blocks.

Fully offline: standard library only (`hashlib`), no real model weights, no network access, no API
keys.

## 1. A synthetic "model weights" artifact and its published reference hash

Real DeepSeek checkpoints are large binary files; here a smaller synthetic byte string stands in for
one. The publisher's SHA-256 hash is computed once and treated as the trusted reference value --
exactly the kind of hash DeepSeek publishes alongside a real model release.

In [1]:
import hashlib

def compute_sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


# Synthetic "model weights" -- deterministic content standing in for a real checkpoint's bytes
genuine_weights = bytes(range(256)) * 1000
print(f"Synthetic artifact size: {len(genuine_weights):,} bytes")

published_hash = compute_sha256(genuine_weights)
print("Publisher-published reference SHA-256:", published_hash)


Synthetic artifact size: 256,000 bytes
Publisher-published reference SHA-256: b57b64b198d5d59ce5a22a9b9f25e72a7d081476d432051aa923f3dbebb90934


## 2. The verify-before-deploy gate

This is the hard-block mechanism Chapter 6 describes: a downloaded artifact is hashed, compared
against the published reference, and only allowed to proceed to deployment on an exact match. A
mismatch is quarantined -- never deployed with a logged warning, never "close enough."

In [2]:
def verify_and_deploy(candidate_weights: bytes, published_hash: str, artifact_name: str) -> bool:
    computed = compute_sha256(candidate_weights)
    if computed == published_hash:
        print(f"[{artifact_name}] Hash MATCH ({computed[:16]}...) -> deployment proceeds.")
        return True
    else:
        print(f"[{artifact_name}] Hash MISMATCH!")
        print(f"    expected: {published_hash}")
        print(f"    computed: {computed}")
        print(f"    -> QUARANTINED. Deployment BLOCKED. Download source flagged for investigation.")
        return False


## 3. Case 1: a clean, genuine download

The candidate artifact is byte-for-byte identical to what the publisher released. Verification
passes and deployment is allowed to proceed.

In [3]:
clean_download = bytes(range(256)) * 1000   # byte-for-byte identical to genuine_weights

deploy_ok = verify_and_deploy(clean_download, published_hash, "deepseek-v3-oncology-clean-download")
assert deploy_ok is True
print("\nOK: a genuine, unmodified download passes verification and is allowed to deploy.")


[deepseek-v3-oncology-clean-download] Hash MATCH (b57b64b198d5d59c...) -> deployment proceeds.

OK: a genuine, unmodified download passes verification and is allowed to deploy.


## 4. Case 2: a tampered checkpoint

A single byte is flipped somewhere deep in the artifact -- the kind of subtle alteration that would
be effectively invisible to casual inspection of a large binary weights file, and that could, in
principle, encode a deliberately altered behavior. The hash check catches it immediately, regardless
of how small the alteration is.

In [4]:
tampered = bytearray(genuine_weights)
tampered[123_456 % len(tampered)] ^= 0xFF   # flip every bit at one byte position
tampered_bytes = bytes(tampered)

assert tampered_bytes != genuine_weights   # confirm we actually altered something
diff_byte_count = sum(1 for a, b in zip(genuine_weights, tampered_bytes) if a != b)
print(f"Bytes altered in the tampered artifact: {diff_byte_count} (out of {len(genuine_weights):,})")

deploy_ok_tampered = verify_and_deploy(tampered_bytes, published_hash, "deepseek-v3-oncology-tampered-download")
assert deploy_ok_tampered is False
print("\nOK: a checkpoint altered by a SINGLE byte out of hundreds of thousands is caught by the "
      "hash mismatch and blocked before it ever reaches a SageMaker endpoint -- exactly the "
      "integrity guarantee a casual 'does it look right' review of a binary weights file could "
      "never provide.")


Bytes altered in the tampered artifact: 1 (out of 256,000)
[deepseek-v3-oncology-tampered-download] Hash MISMATCH!
    expected: b57b64b198d5d59ce5a22a9b9f25e72a7d081476d432051aa923f3dbebb90934
    computed: cb44ae25c06a4ac94180c12342d118d9894a7c0ad8ab15275a289cc1eaa6fd0e
    -> QUARANTINED. Deployment BLOCKED. Download source flagged for investigation.

OK: a checkpoint altered by a SINGLE byte out of hundreds of thousands is caught by the hash mismatch and blocked before it ever reaches a SageMaker endpoint -- exactly the integrity guarantee a casual 'does it look right' review of a binary weights file could never provide.


## 5. Recording the provenance chain

A verification event -- pass or fail -- is recorded permanently against the resulting
`model_version` identifier, so months later any screening decision can be traced back to a specific,
verified, genuine checkpoint (tying directly into notebook 04's `model_version` tagging).

In [5]:
from dataclasses import dataclass
from datetime import datetime, timezone


@dataclass
class ProvenanceRecord:
    model_version: str
    source_url: str
    verified_hash: str
    verified_at: str
    verification_result: str   # "VERIFIED" | "REJECTED_HASH_MISMATCH"


def record_provenance(model_version: str, source_url: str, computed_hash: str, published_hash: str) -> ProvenanceRecord:
    result = "VERIFIED" if computed_hash == published_hash else "REJECTED_HASH_MISMATCH"
    return ProvenanceRecord(
        model_version=model_version,
        source_url=source_url,
        verified_hash=computed_hash,
        verified_at=datetime.now(timezone.utc).isoformat(),
        verification_result=result,
    )


clean_record = record_provenance(
    "deepseek-v3-oncology-2026.01",
    "https://model-hub.example/deepseek-v3-oncology-2026.01",
    compute_sha256(clean_download),
    published_hash,
)
tampered_record = record_provenance(
    "deepseek-v3-oncology-2026.01-UNTRUSTED",
    "https://untrusted-mirror.example/deepseek-v3-oncology-2026.01",
    compute_sha256(tampered_bytes),
    published_hash,
)

print("Provenance ledger:")
print(" ", clean_record)
print(" ", tampered_record)

assert clean_record.verification_result == "VERIFIED"
assert tampered_record.verification_result == "REJECTED_HASH_MISMATCH"
print("\nOK: the provenance ledger records both outcomes permanently -- a reviewer or auditor asking "
      "'what model produced this decision, and how do we know it was genuine' gets a precise, "
      "evidenced answer, not an assumption.")


Provenance ledger:
  ProvenanceRecord(model_version='deepseek-v3-oncology-2026.01', source_url='https://model-hub.example/deepseek-v3-oncology-2026.01', verified_hash='b57b64b198d5d59ce5a22a9b9f25e72a7d081476d432051aa923f3dbebb90934', verified_at='2026-07-28T18:30:38.500328+00:00', verification_result='VERIFIED')
  ProvenanceRecord(model_version='deepseek-v3-oncology-2026.01-UNTRUSTED', source_url='https://untrusted-mirror.example/deepseek-v3-oncology-2026.01', verified_hash='cb44ae25c06a4ac94180c12342d118d9894a7c0ad8ab15275a289cc1eaa6fd0e', verified_at='2026-07-28T18:30:38.500471+00:00', verification_result='REJECTED_HASH_MISMATCH')

OK: the provenance ledger records both outcomes permanently -- a reviewer or auditor asking 'what model produced this decision, and how do we know it was genuine' gets a precise, evidenced answer, not an assumption.


## 6. Tying it back

- The verify-before-deploy gate (Section 2-4) is a **hard block**, not a soft warning: a hash
  mismatch, however small the underlying alteration, stops deployment outright, the same
  content-integrity discipline Chapter 8's error-handling table applies to every other
  correctness-critical failure in this pipeline.
- A single altered byte in a large artifact is enough to be caught (Section 4) -- functional testing
  alone might never surface a subtly tampered checkpoint, but a cryptographic hash comparison catches
  it deterministically, every time.
- The **provenance record** (Section 5) is what makes Chapter 2's regulatory-auditability argument
  operational rather than aspirational: a `model_version` identifier is only as trustworthy as the
  verification chain proving it's genuinely the artifact it claims to be, and that chain has to be
  recorded, not just performed once and forgotten.